In [ ]:
# ==========================================
# BLOCK 1: ENVIRONMENT SETUP & PERSISTENCE
# ==========================================
# @title ⚙️ 1. Setup Environment & Google Drive
# @markdown Connects to Google Drive, sets up paths, logs, and installs dependencies.

DRIVE_WORKSPACE_NAME = "AutoScribe_Workspace" # @param {type:"string"}
SHARED_DRIVE_NAME = "" # @param {type:"string"}

import os
import sys
from datetime import datetime
from google.colab import drive
from IPython.display import clear_output, display
from IPython.utils import capture
import ipywidgets as widgets

# --- Guardrail: Logger Setup ---
def log_event(level, message, print_to_console=True):
    if print_to_console:
        print(message)
    try:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(LOG_FILE, 'a', encoding='utf-8') as f:
            f.write(f"[{timestamp}] [{level}] {message}\n")
    except Exception:
        pass # Silently fail if log file isn't ready yet

def inf(msg, style, wdth):
    inf_btn = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth))
    display(inf_btn)

print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- Path Resolution ---
if SHARED_DRIVE_NAME != "" and os.path.exists(f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"):
    root_path = f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"
else:
    root_path = "/content/drive/MyDrive"

workspace_name = DRIVE_WORKSPACE_NAME.strip() or "AutoScribe_Workspace"
BASE_DIR = f"{root_path}/{workspace_name}"

# --- Set Up Directories & Global Variables ---
CACHE_DIR = os.path.join(BASE_DIR, "cache")
PIP_CACHE = os.path.join(CACHE_DIR, "pip_packages")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
TEMP_DIR = "/content/temp_media"
MARKER_FILE = os.path.join(CACHE_DIR, ".setup_complete")
LOG_FILE = os.path.join(OUTPUT_DIR, "autoscribe_debug.log")

for directory in [PIP_CACHE, OUTPUT_DIR, TEMP_DIR]:
    os.makedirs(directory, exist_ok=True)

# Initialize Log File
log_event("INFO", "=== NEW AUTOSCRIBE SESSION STARTED ===", print_to_console=False)

os.environ["HF_HOME"] = "/content/local_hf_cache"
sys.path.insert(0, PIP_CACHE)

# --- Silent & Fast Installation ---
if not os.path.exists(MARKER_FILE):
    log_event("INFO", "First time setup: Installing dependencies to Drive...")
    with capture.capture_output() as cap:
        !pip install -q --target=$PIP_CACHE yt-dlp faster-whisper ffmpeg-python transformers accelerate torch torchvision
        with open(MARKER_FILE, 'w') as f:
            f.write("Setup complete.")
    log_event("INFO", "Dependencies installed successfully.")
else:
    log_event("INFO", "Dependencies loaded from cache.", print_to_console=False)

# Flag block as complete for future guardrails
BLOCK_1_COMPLETED = True

clear_output()
inf('\u2714 Workspace Ready & Logging Active', 'success', '350px')

In [ ]:
# ==========================================
# BLOCK 2: INGESTION & ROUTING
# ==========================================
# @title 📥 2. Source Configuration & Routing
# @markdown Select your media source. Invalid downloads are logged and skipped.

SOURCE_TYPE = "URL" # @param ["URL", "Google_Drive", "Local_Upload"]
MEDIA_INPUT = "https://youtube.com/playlist?list=..." # @param {type:"string"}
WHISPER_MODE = "Auto" # @param ["On", "Off", "Auto"]
VISION_FALLBACK = True # @param {type:"boolean"}

# --- Guardrail: State Validation ---
if 'BLOCK_1_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Workspace not initialized. Please run Block 1 first.")

import yt_dlp
import subprocess
import shutil
from google.colab import files

routing_queue = []

def analyze_and_route(file_path, title, video_id):
    log_event("INFO", f"Analyzing audio for: {title}...")
    command = ["ffmpeg", "-i", file_path, "-af", "silencedetect=noise=-50dB:d=2", "-f", "null", "-"]
    requires_vision = False
    
    try:
        result = subprocess.run(command, stderr=subprocess.PIPE, text=True)
        is_silent = "silence_start" in result.stderr or "Audio:" not in result.stderr
        
        if WHISPER_MODE == "Auto":
            if is_silent and VISION_FALLBACK:
                requires_vision = True
                log_event("INFO", "[ROUTING]: Silence detected. Tagged for Vision Processing.")
            else:
                log_event("INFO", "[ROUTING]: Audio intact. Tagged for Whisper.")
        elif WHISPER_MODE == "Off" and VISION_FALLBACK:
            requires_vision = True
            log_event("INFO", "[ROUTING]: Whisper forced OFF. Tagged for Vision Processing.")
            
    except Exception as e:
        log_event("ERROR", f"Audio analysis failed for {title}: {e}. Fallback to Vision.")
        requires_vision = True

    routing_queue.append({
        'id': video_id, 'title': title, 'path': file_path, 'use_vision': requires_vision
    })

log_event("INFO", f"Processing source type: {SOURCE_TYPE}")

if SOURCE_TYPE == "Local_Upload":
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest_path = os.path.join(TEMP_DIR, filename)
        shutil.move(filename, dest_path)
        analyze_and_route(dest_path, filename, filename.split('.')[0])

elif SOURCE_TYPE == "Google_Drive":
    if os.path.exists(MEDIA_INPUT):
        filename = os.path.basename(MEDIA_INPUT)
        dest_path = os.path.join(TEMP_DIR, filename)
        shutil.copy2(MEDIA_INPUT, dest_path)
        analyze_and_route(dest_path, filename, filename.split('.')[0])
    else:
        log_event("ERROR", "File not found on Google Drive.")

elif SOURCE_TYPE == "URL":
    ydl_opts_meta = {
        'extract_flat': 'in_playlist',
        'quiet': True,
        'js_runtimes': {'nodejs': {}, 'node': {}}
    }
    
    with yt_dlp.YoutubeDL(ydl_opts_meta) as ydl:
        try:
            info = ydl.extract_info(MEDIA_INPUT, download=False)
            entries = info['entries'] if 'entries' in info else [info]
            
            for entry in entries:
                v_id = entry['id']
                title = entry.get('title', v_id)
                url = entry.get('url', MEDIA_INPUT)
                
                log_event("INFO", f"--- Downloading: {title} ---")
                
                ydl_opts_dl = {
                    'format': 'bestaudio/best',
                    'outtmpl': f'{TEMP_DIR}/{v_id}.%(ext)s',
                    'trim_file_name': 200,
                    'quiet': True,
                    'js_runtimes': {'nodejs': {}, 'node': {}}
                }
                
                with yt_dlp.YoutubeDL(ydl_opts_dl) as ydl_dl:
                    try:
                        dl_info = ydl_dl.extract_info(url, download=True)
                        local_file = ydl_dl.prepare_filename(dl_info)
                        analyze_and_route(local_file, title, v_id)
                    except Exception as e:
                        log_event("ERROR", f"Failed to download {title}: {e}")
        except Exception as e:
            log_event("ERROR", f"Failed to parse URL/Playlist: {e}")

# Flag block as complete
BLOCK_2_COMPLETED = True
log_event("INFO", f"✅ Block 2 Complete. {len(routing_queue)} item(s) in queue.")

In [ ]:
# ==========================================
# BLOCK 3: AI PROCESSING (Whisper & Moondream2)
# ==========================================
# @title 🧠 3. Execute AI Transcription & Vision Analysis
# @markdown Ensure your `HF_TOKEN` is set in Colab Secrets before running.

import gc
import torch
import subprocess
from PIL import Image

# --- Guardrail: State Validation ---
if 'BLOCK_2_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: System memory lost or queue is empty. Please run Block 1 and Block 2 first.")
if not routing_queue:
    raise ValueError("🚨 ERROR: Routing queue is empty. No valid media was processed in Block 2.")

# Check for required modules
try:
    from faster_whisper import WhisperModel
except ImportError:
    raise RuntimeError("🚨 ERROR: AI Libraries not found. sys.path was lost. Run Block 1 to restore paths.")

# --- Guardrail: HF Token Validation ---
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
    log_event("INFO", "✅ Hugging Face token authenticated.")
except Exception:
    error_msg = "🚨 CRITICAL ERROR: 'HF_TOKEN' not found in Colab Secrets. Models cannot be downloaded."
    log_event("ERROR", error_msg)
    raise PermissionError(error_msg)

LOCAL_MODEL_CACHE = "/content/local_hf_cache"
os.makedirs(LOCAL_MODEL_CACHE, exist_ok=True)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

transcription_results = []
audio_tasks = [t for t in routing_queue if not t['use_vision']]
vision_tasks = [t for t in routing_queue if t['use_vision']]

# ------------------------------------------
# STAGE 1: AUDIO PROCESSING
# ------------------------------------------
if audio_tasks:
    log_event("INFO", "\n--- Loading Whisper Model (large-v3) ---")
    try:
        whisper_model = WhisperModel("large-v3", device="cuda", compute_type="float16", download_root=LOCAL_MODEL_CACHE)
        
        for task in audio_tasks:
            log_event("INFO", f"🎙️ Transcribing Audio: {task['title']}")
            try:
                segments, info = whisper_model.transcribe(task['path'], beam_size=5, language="sv", condition_on_previous_text=False)
                text_output = []
                for segment in segments:
                    start_m, start_s = divmod(int(segment.start), 60)
                    text_output.append(f"[{start_m:02d}:{start_s:02d}] {segment.text.strip()}")
                
                transcription_results.append({
                    'title': task['title'],
                    'content': "\n".join(text_output),
                    'type': 'Audio Transcription'
                })
            except Exception as e:
                log_event("ERROR", f"Whisper failed for {task['title']}: {e}")

    except Exception as e:
        log_event("ERROR", f"Failed to load Whisper Model: {e}")
        
    finally:
        log_event("INFO", "Flushing Whisper from GPU memory...")
        if 'whisper_model' in locals():
            del whisper_model
        gc.collect()
        torch.cuda.empty_cache()

# ------------------------------------------
# STAGE 2: VISION PROCESSING
# ------------------------------------------
if vision_tasks:
    log_event("INFO", "\n--- Loading Moondream2 Vision Model ---")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        model_id = "vikhyatk/moondream2"
        tokenizer = AutoTokenizer.from_pretrained(model_id, revision="2024-04-02", cache_dir=LOCAL_MODEL_CACHE)
        moondream = AutoModelForCausalLM.from_pretrained(
            model_id, trust_remote_code=True, revision="2024-04-02", cache_dir=LOCAL_MODEL_CACHE
        ).to("cuda")
        moondream.eval()

        for task in vision_tasks:
            log_event("INFO", f"👁️ Processing Vision: {task['title']}")
            frame_dir = os.path.join(TEMP_DIR, f"frames_{task['id']}")
            os.makedirs(frame_dir, exist_ok=True)
            
            subprocess.run([
                "ffmpeg", "-i", task['path'], "-vf", "fps=1/30", 
                f"{frame_dir}/frame_%04d.jpg", "-hide_banner", "-loglevel", "error"
            ])
            
            frames = sorted(os.listdir(frame_dir))
            text_output = []
            
            for i, frame_file in enumerate(frames):
                img_path = os.path.join(frame_dir, frame_file)
                try:
                    image = Image.open(img_path)
                    enc_image = moondream.encode_image(image)
                    answer = moondream.answer_question(
                        enc_image, 
                        "Describe the key action, text, or visual information in this frame concisely.", 
                        tokenizer
                    )
                    time_sec = i * 30
                    start_m, start_s = divmod(time_sec, 60)
                    text_output.append(f"[{start_m:02d}:{start_s:02d}] {answer}")
                except Exception as e:
                    log_event("WARNING", f"Skipping frame {frame_file}: {e}")
                    
            transcription_results.append({
                'title': task['title'],
                'content': "\n".join(text_output),
                'type': 'Visual Analysis'
            })

    except Exception as e:
        log_event("ERROR", f"Failed to load/process Vision Model: {e}")
        
    finally:
        log_event("INFO", "Flushing Moondream2 from GPU memory...")
        if 'moondream' in locals():
            del moondream
        gc.collect()
        torch.cuda.empty_cache()

BLOCK_3_COMPLETED = True
log_event("INFO", f"✅ Block 3 Complete. Processed {len(transcription_results)} items.")

In [ ]:
# ==========================================
# BLOCK 4: FORMATTING, EXPORT & CLEANUP
# ==========================================
# @title 📝 4. Export to Markdown & Auto-Shutdown
# @markdown Formats the text, saves to Drive, and safely disconnects the runtime.

import shutil
from google.colab import runtime

# --- Guardrail: State Validation ---
if 'BLOCK_3_COMPLETED' not in locals() or 'transcription_results' not in locals():
    raise RuntimeError("🚨 ERROR: Cannot export. Block 3 must run successfully first.")
if not transcription_results:
    log_event("WARNING", "No results to export. Skipping export phase.")
else:
    MAX_WORDS_PER_FILE = 400000
    PROJECT_NAME = "AutoScribe_Export"
    
    log_event("INFO", "--- Formatting Outputs for NotebookLM ---")
    
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    export_dir = os.path.join(OUTPUT_DIR, f"{timestamp}_{PROJECT_NAME}")
    os.makedirs(export_dir, exist_ok=True)
    
    current_word_count = 0
    file_index = 1
    output_content = f"# {PROJECT_NAME} - Part {file_index}\n\n"
    
    def save_markdown(index, content):
        filepath = os.path.join(export_dir, f"{PROJECT_NAME}_Part_{index}.md")
        try:
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(content)
            log_event("INFO", f"📁 Saved: {filepath}")
        except Exception as e:
            log_event("ERROR", f"Failed to save {filepath}: {e}")
    
    for result in transcription_results:
        title = result['title']
        content = result['content']
        source_type = result['type']
        
        block_text = f"## --- START SOURCE: {title} ---\n"
        block_text += f"**Analysis Type:** {source_type}\n\n"
        block_text += f"{content}\n\n"
        block_text += f"## --- END SOURCE: {title} ---\n\n"
        
        words_in_block = len(block_text.split())
        
        if current_word_count + words_in_block > MAX_WORDS_PER_FILE:
            save_markdown(file_index, output_content)
            file_index += 1
            output_content = f"# {PROJECT_NAME} - Part {file_index}\n\n"
            output_content += f"> *[CONTINUED FROM PART {file_index-1}]*\n\n"
            current_word_count = 0
            
        output_content += block_text
        current_word_count += words_in_block
    
    if current_word_count > 0:
        save_markdown(file_index, output_content)

# ------------------------------------------
# CLEANUP & SHUTDOWN
# ------------------------------------------
log_event("INFO", "--- Cleaning up temporary files ---")
try:
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
        log_event("INFO", "🗑️ Ephemeral media storage cleared.")
except Exception as e:
    log_event("WARNING", f"Cleanup warning: {e}")

log_event("INFO", "🎉 All processing complete. Disconnecting runtime.")

# Disconnect the runtime
runtime.unassign()